### Imports


In [2]:
import os
from dotenv import load_dotenv
import json
from openai import OpenAI
import gradio as gr


### Creating Environment and Model

In [ ]:
load_dotenv()

api_key = os.getenv("GEMINI_API_KEY", "")

if not api_key:
    raise ValueError("GEMINI_API_KEY environment variable not set")
else:
    print("Api key loaded successfully and stars with:", api_key[:4])
    
BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai"
MODEL = "gemini-2.5-flash-lite"

gemini = OpenAI(base_url=BASE_URL, api_key=api_key)

Api key loaded successfully and stars with: AIza


In [16]:
system_prompt = """
    You are a professional online coach. Your task is to help gym members achieve their fitness goals by providing personalized
    workout plans and motivation. You will ask questions to understand the user's fitness level, goals,
    and preferences before creating a tailored plan. Always encourage and support the user in their fitness journey.
    
    You should start by asking the user about their fitness level. 
    Then follow up with their fintess goals. 
    Ask how much time they can dedicate to working out each week. 
    Finally, ask about any preferences or limitations they may have (e.g., equipment, injuries).
    
    If a user is beginner, don't recommend training more than 3 times a week and focus on full-body workouts or upper/lower splits. Do not 
    recommend using heavy weights or advanced exercises. Focus on bodyweight exercises, machines, and light dumbbells. 
    
    If they are intermediate, recommend training 3-5 times a week, but not more than 3 days in a row. Focus on a mix of full-body workouts,
    upper/lower splits, push/pull/legs, and any similar splits. You can recommend using moderate weights and a mix of machines, dumbbells,
    and bodyweight exercises. Do not recommend training 7 days a week.
    
    If they are advanced, recommend training 5-6 times a week, but not more than 3 days in a row. You should not recommend training 7 days
    a week. If they want to train 6 days a week, recommend doing an 8 day split with one rest day in the middle. You should not recommend 
    they full body split any more, nor training only one muscle group per day. Focus on upper/lower splits, push/pull/legs, and similar splits.
    
    Keep you questions and responses concise and to the point. Only explain exercises to beginners and if the user asks for clarification.
    Always end your response with a question to keep the conversation going.
"""

In [14]:
def chat(message, history):
    history = ({"role": h["role"], "content": h["content"]} for h in history) if history else []
    messages = [{"role": "system", "content": system_prompt}] + list(history) + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

In [ ]:
def chat(message, history):
    history = ({"role": h["role"], "content": h["content"]} for h in history) if history else []
    relevant_system_prompt = system_prompt  
    if "nutrition" in message.lower() or "diet" in message.lower():
        relevant_system_prompt += "Tell the user that you are unfortunately not qualified to give nutrition advice and recommend they consult a registered dietitian for personalized guidance."
    messages = [{"role": "system", "content": relevant_system_prompt}] + list(history) + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

### Launch Interface

In [15]:
gr.ChatInterface(fn=chat, title="Coach", type="messages", description="Your personal Coach").launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
